# Preprocessing Transactions

This file preprocesses the transaction data 

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import re
import glob
import math
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DateType, DoubleType
from datetime import datetime, timedelta

In [3]:
spark = (
    SparkSession.builder.appName('Transactions Preprocess')
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '2g')
    .getOrCreate()
)


24/10/16 16:31:57 WARN Utils: Your hostname, Alans-MacBook-Air-4.local resolves to a loopback address: 127.0.0.1; using 192.168.0.52 instead (on interface en0)
24/10/16 16:31:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/10/16 16:31:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/10/16 16:31:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Join all transaction data into one dataframe.

In [ ]:
transactions = spark.read.parquet(".././data/landing/transactions/transactions_20210228_20210827_snapshot")

transactions.show()

+-------+------------+------------------+--------------------+--------------+
|user_id|merchant_abn|      dollar_value|            order_id|order_datetime|
+-------+------------+------------------+--------------------+--------------+
|  18478| 62191208634|63.255848959735246|949a63c8-29f7-4ab...|    2021-08-20|
|      2| 15549624934| 130.3505283105634|6a84c3cf-612a-457...|    2021-08-20|
|  18479| 64403598239|120.15860593212783|b10dcc33-e53f-425...|    2021-08-20|
|      3| 60956456424| 136.6785200286976|0f09c5a5-784e-447...|    2021-08-20|
|  18479| 94493496784| 72.96316578355305|f6c78c1a-4600-4c5...|    2021-08-20|
|      3| 76819856970|  448.529684285612|5ace6a24-cdf0-4aa...|    2021-08-20|
|  18479| 67609108741|  86.4040605836911|d0e180f0-cb06-42a...|    2021-08-20|
|      3| 34096466752| 301.5793450525113|6fb1ff48-24bb-4f9...|    2021-08-20|
|  18482| 70501974849| 68.75486276223054|8505fb33-b69a-412...|    2021-08-20|
|      4| 49891706470| 48.89796461900801|ed11e477-b09f-4ae...|  

In [5]:
transactions.write.mode('overwrite').parquet('.././data/landing/transactions/total_transactions')

In [6]:
transactions = spark.read.parquet(".././data/landing/transactions/total_transactions")
transactions.show(truncate = False)

+-------+------------+------------------+------------------------------------+--------------+
|user_id|merchant_abn|dollar_value      |order_id                            |order_datetime|
+-------+------------+------------------+------------------------------------+--------------+
|14935  |79417999332 |136.06570809815838|23acbb7b-cf98-4580-9775-86b8e0a2bd88|2021-11-26    |
|1      |46451548968 |72.61581642788431 |76bab304-fa2d-4004-8179-8638b56a873e|2021-11-26    |
|14936  |89518629617 |3.0783487174439297|a2ae446a-2959-41c4-81fd-a30c1efbde0c|2021-11-26    |
|1      |49167531725 |51.58228625503599 |7080c274-17f7-4ccf-be22-1d33a45f7f81|2021-11-26    |
|14936  |31101120643 |25.228114942417797|8e301c0f-06ab-45cb-a573-47c8e9b95423|2021-11-26    |
|2      |67978471888 |691.5028234458998 |0380e9ad-b0e8-4203-a74b-e4ad3504d643|2021-11-26    |
|14936  |60956456424 |102.13952056640888|5ac3da9c-5147-4524-8efb-0d312cfe6748|2021-11-26    |
|2      |47644196714 |644.5220654863093 |4e368e44-86f8-4dee-

In [9]:
transactions.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- merchant_abn: long (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)



Ensure all datatypes are consistent across tables.

In [10]:
transactions = transactions.withColumn('user_id', F.col('user_id').cast(StringType()))

In [11]:
transactions = transactions.withColumn('merchant_abn', F.col('merchant_abn').cast(StringType()))

In [12]:
transactions.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- merchant_abn: string (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)



Check user_id is valid.

In [13]:
consumer_details = spark.read.parquet(".././data/curated/consumer_details")
consumer_details.show(truncate = False)

+-------+-----------+
|user_id|consumer_id|
+-------+-----------+
|1      |1195503    |
|2      |179208     |
|3      |1194530    |
|4      |154128     |
|5      |712975     |
|6      |407340     |
|7      |511685     |
|8      |448088     |
|9      |650435     |
|10     |1058499    |
|11     |428325     |
|12     |1494640    |
|13     |1146717    |
|14     |1343547    |
|15     |1463076    |
|16     |1356405    |
|17     |1331093    |
|18     |80965      |
|19     |1226530    |
|20     |1390367    |
+-------+-----------+
only showing top 20 rows



In [14]:
userid = consumer_details.select(F.col('user_id')).distinct()
userid = userid.toPandas()
userid = userid['user_id'].tolist()

In [15]:
transacid = transactions.select(F.col('user_id')).distinct()
transacid = transacid.toPandas()
transacid = transacid['user_id'].tolist()

In [16]:
transactions.count()

14195505

In [17]:
print(len(set(transacid).difference(userid)))

0


In [18]:
print(set(transacid).difference(userid))

set()


Check merchant_abn is valid.

In [19]:
tbl_merchants = spark.read.parquet(".././data/curated/tbl_merchants")
tbl_merchants.show(truncate = False)


+------------------------------------+------------+----------------------------------------------------------------+----+---------+
|merchant_name                       |merchant_abn|tags                                                            |type|take_rate|
+------------------------------------+------------+----------------------------------------------------------------+----+---------+
|Felis Limited                       |10023283211 |furniture, home furnishings, equipment, manufacturers           |e   |0.18     |
|Arcu Ac Orci Corporation            |10142254217 |cable, satellite, other pay television, radio                   |b   |4.22     |
|Nunc Sed Company                    |10165489824 |jewelry, watch, clock, silverware                               |b   |4.4      |
|Ultricies Dignissim Lacus Foundation|10187291046 |watch, clock, jewelry repair                                    |b   |3.29     |
|Enim Condimentum PC                 |10192359162 |music, musical instrument

In [20]:
tbl_merchants = tbl_merchants.withColumn('merchant_abn',F.col('merchant_abn').cast(StringType()) ) 

Find any merchant_abn's that have transactions for whom data has not been found.

In [21]:
merchants_abn = tbl_merchants.select(F.col('merchant_abn')).distinct()
merchants_abn = merchants_abn.toPandas()
merchants_abn = merchants_abn['merchant_abn'].tolist()

In [22]:
transacabn = transactions.select(F.col('merchant_abn')).distinct()
transacabn = transacabn.toPandas()
transacabn = transacabn['merchant_abn'].tolist()

In [23]:
transactions.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- merchant_abn: string (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)



In [24]:
tbl_merchants.printSchema()

root
 |-- merchant_name: string (nullable = true)
 |-- merchant_abn: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- type: string (nullable = true)
 |-- take_rate: double (nullable = true)



In [25]:
tbl_merchants.printSchema()

root
 |-- merchant_name: string (nullable = true)
 |-- merchant_abn: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- type: string (nullable = true)
 |-- take_rate: double (nullable = true)



In [26]:
print(len(set(transacabn).difference(merchants_abn)))

396


In [27]:
print(set(transacabn).difference(merchants_abn))

{'83893827922', '79123236887', '96129954993', '71091910300', '75342681786', '57241240228', '32297823633', '84500933183', '35601926858', '93360876880', '48032257165', '94100550074', '17135205181', '45925655949', '94311056026', '32649670245', '67254399155', '13177076095', '86398826270', '92396103355', '74448581190', '62321559724', '32831345323', '29323795999', '25810287809', '68102957262', '57684437046', '95390063681', '44881138767', '23495610951', '93559143257', '32461318592', '31507950402', '81146325646', '48452300785', '58830373879', '24301977828', '77126808895', '20562405782', '29227046551', '92442801816', '67202032418', '88984392677', '67330176930', '33604812025', '60892515210', '56703883889', '67214015867', '52985515941', '71836374493', '82999039227', '51914572250', '42211422632', '90628017098', '82729532791', '54082851220', '25569327276', '31761538646', '51776659549', '87921002735', '98770741241', '40576714979', '22119749782', '61333486380', '89738742187', '74777410897', '52447384

In [28]:
lst = list(set(transacabn).difference(merchants_abn))

In [29]:
dct = {'merchants_abn' : lst}
df = pd.DataFrame(dct)
     
# saving the dataframe
df.to_csv('missingabn.csv')

In [30]:
tbl_merchants.filter(F.col('merchant_abn') == '29836312398').show()

+-------------+------------+----+----+---------+
|merchant_name|merchant_abn|tags|type|take_rate|
+-------------+------------+----+----+---------+
+-------------+------------+----+----+---------+



In [31]:
transactions.filter(F.col('merchant_abn') == '29836312398').show()

+-------+------------+------------------+--------------------+--------------+
|user_id|merchant_abn|      dollar_value|            order_id|order_datetime|
+-------+------------+------------------+--------------------+--------------+
|   8571| 29836312398| 2945.505025734725|b627ed8d-262b-4cd...|    2021-12-11|
|  12517| 29836312398|1543.1517906741005|e32c9ad9-ff31-411...|    2022-10-23|
|   1620| 29836312398|   6258.0924425422|fc4c3ccd-dd22-44a...|    2022-10-26|
|  12615| 29836312398|3813.1665926469286|9e572064-6fde-416...|    2022-08-23|
|  21939| 29836312398| 3899.466796707913|00d9d0d2-7b23-403...|    2022-05-29|
|     69| 29836312398| 4411.945163042472|14f63cf5-d8a0-48f...|    2022-07-28|
|  16922| 29836312398|3448.2403740176865|a38da7f6-4da4-4a3...|    2022-06-03|
|   4687| 29836312398|3807.7129245420906|989ac26e-1e23-424...|    2022-06-07|
|  12991| 29836312398| 2072.397078128837|0368c646-6390-4d8...|    2021-11-07|
|  19145| 29836312398| 831.8509457977442|03816e55-0beb-4f6...|  

check dollar value

In [32]:
transactions.filter(F.col('dollar_value') < 0).show()

+-------+------------+------------+--------+--------------+
|user_id|merchant_abn|dollar_value|order_id|order_datetime|
+-------+------------+------------+--------+--------------+
+-------+------------+------------+--------+--------------+



check null values

In [33]:
transactions.filter(F.isnull('dollar_value')).show()

+-------+------------+------------+--------+--------------+
|user_id|merchant_abn|dollar_value|order_id|order_datetime|
+-------+------------+------------+--------+--------------+
+-------+------------+------------+--------+--------------+



In [34]:
transactions.filter(F.isnull('order_id')).show()

+-------+------------+------------+--------+--------------+
|user_id|merchant_abn|dollar_value|order_id|order_datetime|
+-------+------------+------------+--------+--------------+
+-------+------------+------------+--------+--------------+



In [35]:
transactions.filter(F.isnull('user_id')).show()

+-------+------------+------------+--------+--------------+
|user_id|merchant_abn|dollar_value|order_id|order_datetime|
+-------+------------+------------+--------+--------------+
+-------+------------+------------+--------+--------------+



In [36]:
transactions.filter(F.isnull('order_datetime')).show()

+-------+------------+------------+--------+--------------+
|user_id|merchant_abn|dollar_value|order_id|order_datetime|
+-------+------------+------------+--------+--------------+
+-------+------------+------------+--------+--------------+



In [37]:
transactions.filter(F.isnull('merchant_abn')).show()

+-------+------------+------------+--------+--------------+
|user_id|merchant_abn|dollar_value|order_id|order_datetime|
+-------+------------+------------+--------+--------------+
+-------+------------+------------+--------+--------------+



In [38]:
transactions.filter(F.isnan('dollar_value')).show()

+-------+------------+------------+--------+--------------+
|user_id|merchant_abn|dollar_value|order_id|order_datetime|
+-------+------------+------------+--------+--------------+
+-------+------------+------------+--------+--------------+



Removing merchant_abn not in tbl_merchants

In [39]:
lst

['83893827922',
 '79123236887',
 '96129954993',
 '71091910300',
 '75342681786',
 '57241240228',
 '32297823633',
 '84500933183',
 '35601926858',
 '93360876880',
 '48032257165',
 '94100550074',
 '17135205181',
 '45925655949',
 '94311056026',
 '32649670245',
 '67254399155',
 '13177076095',
 '86398826270',
 '92396103355',
 '74448581190',
 '62321559724',
 '32831345323',
 '29323795999',
 '25810287809',
 '68102957262',
 '57684437046',
 '95390063681',
 '44881138767',
 '23495610951',
 '93559143257',
 '32461318592',
 '31507950402',
 '81146325646',
 '48452300785',
 '58830373879',
 '24301977828',
 '77126808895',
 '20562405782',
 '29227046551',
 '92442801816',
 '67202032418',
 '88984392677',
 '67330176930',
 '33604812025',
 '60892515210',
 '56703883889',
 '67214015867',
 '52985515941',
 '71836374493',
 '82999039227',
 '51914572250',
 '42211422632',
 '90628017098',
 '82729532791',
 '54082851220',
 '25569327276',
 '31761538646',
 '51776659549',
 '87921002735',
 '98770741241',
 '40576714979',
 '221197

In [40]:
transactions.filter(F.col('merchant_abn') != '29836312398').count()

14195484

In [41]:
transactions = transactions.filter(F.col('merchant_abn') != '29836312398')

In [42]:
transactions.count()

14195484

In [43]:
transactions = transactions.filter(~transactions['merchant_abn'].isin(lst))


In [44]:
transactions.count()

13614675

In [45]:
transacabn1 = transactions.select(F.col('merchant_abn')).distinct()
transacabn1 = transacabn1.toPandas()
transacabn1 = transacabn1['merchant_abn'].tolist()

In [46]:
transactions.count()

13614675

In [47]:
print(len(set(transacabn1).difference(merchants_abn)))

0


In [48]:
transactions.count()

13614675

In [49]:
transactions.write.mode('overwrite').parquet('.././data/curated/total_transactions')

The number of merchants we have is 4026.

In [50]:
tbl_merchants.count()

4026